# Pilas y Colas 

Estas estructuras se conocen como "Tipos de Datos Abstractos" (ADT) debido a que definen un comportamiento lógico riguroso sobre cómo se almacenan y recuperan los elementos, independientemente de su implementación física subyacente.

1. ***Pilas (Stacks)***: Se rigen por el principio *LIFO* (Last-In, First-Out), donde el último elemento en entrar es el primero en salir. Es análogo a una pila de platos: solo puedes interactuar con el plato superior de manera directa.
   
2. ***Colas (Queues)***: Siguen el principio *FIFO* (First-In, First-Out), donde el primer elemento en llegar es el primero en ser atendido. Es análogo a una fila en un banco.

## Abstracción de Pilas en C++
Para implementar estas estructuras con rigor profesional, utilizaremos clases abstractas o interfaces. Esto permite separar la especificación (qué hace la estructura) de la implementación (cómo lo hace, ya sea estática o dinámica).

A continuación, se presenta el código de una interfaz base para una Pila utilizando Templates de C++, lo cual garantiza que nuestra estructura sea genérica y pueda manejar cualquier tipo de dato con seguridad de tipos (type-safety)

```cpp
#include <iostream>
#include <stdexcept>

/**
 * @brief Interfaz Abstracta para una Pila (Stack).
 * Define el contrato que cualquier implementación (estática o dinámica) debe seguir.
 */
template <typename T>
class IStack {
public:
    virtual ~IStack() {}

    // Operaciones fundamentales
    virtual void push(const T& element) = 0; // O(1)
    virtual T pop() = 0;                     // O(1)
    virtual T top() const = 0;               // O(1)
    
    // Consultas de estado
    virtual bool isEmpty() const = 0;
    virtual size_t getSize() const = 0;
};

In [1]:
#include <iostream>
#include <stdexcept>

/**
 * @brief Interfaz Abstracta para una Pila (Stack).
 * Define el contrato que cualquier implementación (estática o dinámica) debe seguir.
 */
template <typename T>
class IStack {
public:
    virtual ~IStack() {}

    // Operaciones fundamentales
    virtual void push(const T& element) = 0; // O(1)
    virtual T pop() = 0;                     // O(1)
    virtual T top() const = 0;               // O(1)
    
    // Consultas de estado
    virtual bool isEmpty() const = 0;
    virtual size_t getSize() const = 0;
};

# Validacion de sintaxis
Para ilustrar el poder de la abstracción LIFO, analizaremos un problema clásico en la construcción de compiladores: la validación de paréntesis balanceados. Este algoritmo simula cómo un analizador sintáctico (parser) verifica la estructura de bloques de código.

In [2]:
#include <iostream>
#include <stack>
#include <string_view>
#include <unordered_map>

/**
 * @brief Valida si una cadena de código tiene sus bloques delimitadores balanceados.
 * Utiliza semántica LIFO para asegurar que el último bloque abierto sea el primero en cerrarse.
 * * @param code Vista de solo lectura de la cadena (evita copias innecesarias).
 * @return true si la sintaxis es válida, false en caso de violación estructural.
 */
bool isValidSyntax(std::string_view code) {
    std::stack<char> parserStack;
    
    // Tabla de búsqueda hash para resolución O(1) de pares de delimitadores
    const std::unordered_map<char, char> matchingBrackets = {
        {')', '('}, 
        {']', '['}, 
        {'}', '{'}
    };

    for (char ch : code) {
        // 1. Condición de Apertura: Apilar (Push)
        if (ch == '(' || ch == '[' || ch == '{') {
            parserStack.push(ch);
        } 
        // 2. Condición de Cierre: Verificar y Desapilar (Pop)
        else if (matchingBrackets.count(ch)) {
            // Stack Underflow estructural o discordancia LIFO
            if (parserStack.empty() || parserStack.top() != matchingBrackets.at(ch)) {
                return false; 
            }
            parserStack.pop();
        }
    }
    
    // Si la pila está vacía al final, todos los bloques fueron cerrados correctamente.
    return parserStack.empty();
}

void ejemplo_01() {
    std::string_view validCode = "int main() { if(true) [ return 0; ] }";
    std::string_view invalidCode = "void func() { ( }";

    std::cout << "Sintaxis 1: " << (isValidSyntax(validCode) ? "Valida" : "Invalida") << "\n";
    std::cout << "Sintaxis 2: " << (isValidSyntax(invalidCode) ? "Valida" : "Invalida") << "\n";
}

ejemplo_01();

Sintaxis 1: Valida
Sintaxis 2: Invalida


## Implementación de la Pila Estática
Iniciamos el núcleo técnico de la sesión. La implementación más directa de una Pila utiliza un arreglo unidimensional. En esta arquitectura, los elementos se almacenan en bloques de memoria contigua, lo que proporciona una excelente localidad de caché espacial, maximizando el rendimiento a nivel de hardware.

1. ***Diseño (Pseudocódigo)***
El estado interno de una pila estática requiere tres variables:

    * `arreglo`: La estructura de almacenamiento.

    * `capacidad`: El tamaño máximo predefinido.

    * `top`: Un índice entero que apunta a la cima de la pila. Se inicializa en -1 para representar una pila vacía.
  
***Algoritmos***

```
1 PUSH(Pila, elemento):
2    Si Pila.top == Pila.capacidad - 1 entonces
3        Lanzar Excepción "Stack Overflow" (Desbordamiento)
4    
5    Pila.top = Pila.top + 1
6    Pila.arreglo[Pila.top] = elemento
```

```
1 POP(Pila):
2    Si Pila.top == -1 entonces
3        Lanzar Excepción "Stack Underflow" (Subdesbordamiento)
4    
5    valor = Pila.arreglo[Pila.top]
6    Pila.top = Pila.top - 1
7    Retornar valor
```

In [3]:
#include <iostream>
#include <stdexcept>

// Se asume la existencia de la interfaz IStack<T> 

template <typename T>
class ArrayStack : public IStack<T> {
private:
    T* array;           // Puntero al bloque de memoria contigua
    size_t capacity;    // Límite máximo de elementos
    int topIndex;       // Apuntador lógico a la cima

public:
    // Constructor: Asignación dinámica de memoria contigua
    explicit ArrayStack(size_t cap = 100) : capacity(cap), topIndex(-1) {
        array = new T[capacity];
    }

    // Destructor: Prevención estricta de memory leaks
    ~ArrayStack() override {
        delete[] array;
    }

    void push(const T& element) override {
        // Validación de límite superior
        if (topIndex == static_cast<int>(capacity) - 1) {
            throw std::overflow_error("Stack Overflow: Capacidad excedida.");
        }
        // Pre-incremento de índice y asignación O(1)
        array[++topIndex] = element;
    }

    T pop() override {
        // Validación de límite inferior
        if (isEmpty()) {
            throw std::underflow_error("Stack Underflow: Pila vacia.");
        }
        // Retorno de valor y post-decremento lógico O(1)
        return array[topIndex--];
    }

    T top() const override {
        if (isEmpty()) {
            throw std::underflow_error("Pila vacia: No hay elementos.");
        }
        return array[topIndex];
    }

    bool isEmpty() const override {
        return topIndex == -1;
    }

    size_t getSize() const override {
        return topIndex + 1;
    }
};

# La Pila Dinámica

Habiendo identificado el cuello de botella espacial de la implementación estática, introducimos la arquitectura dinámica. Una Pila basada en Listas Enlazadas (Linked Stack) asigna memoria de manera individual para cada elemento en el heap (montón) solo cuando es estrictamente necesario.

## Diseño (Pseudocódigo)
El estado interno requiere definir un `Nodo` compuesto por dos campos: el `valor` y un puntero siguiente (`next`) hacia el nodo subyacente. La estructura principal solo necesita mantener un puntero top (cima) que apunte al primer nodo.

***Algortimos***
```
 1 PUSH(Pila, elemento):
 2    // 1. Asignar memoria para un nuevo nodo
 3    nuevo_nodo = crear_nodo()
 4    nuevo_nodo.valor = elemento
 5    
 6    // 2. Enlazar el nuevo nodo con la cima actual
 7    nuevo_nodo.siguiente = Pila.top
 8    
 9    // 3. Actualizar el puntero principal
10    Pila.top = nuevo_nodo
```

```
 1 POP(Pila):
 2   Si Pila.top es NULO entonces
 3       Lanzar Excepción "Stack Underflow"
 4   
 5   // 1. Aislar el nodo en la cima
 6   nodo_a_eliminar = Pila.top
 7   valor_extraido = nodo_a_eliminar.valor
 8   
 9   // 2. Actualizar la cima al siguiente nodo
10    Pila.top = Pila.top.siguiente
11    
12    // 3. Liberar la memoria del nodo aislado
13    liberar_memoria(nodo_a_eliminar)
14    
15    Retornar valor_extraido
```


In [4]:
// Se asume la existencia de la interfaz IStack<T>

template <typename T>
class LinkedStack : public IStack<T> {
private:
    // Estructura interna del Nodo
    struct Node {
        T data;
        Node* next;
        // Constructor del nodo para inicialización directa
        Node(const T& val, Node* nextNode = nullptr) : data(val), next(nextNode) {}
    };

    Node* topNode;      // Puntero al elemento en la cima
    size_t currentSize; // Mantenemos el tamaño para consultas O(1)

public:
    // Constructor: Inicializa la pila vacía
    LinkedStack() : topNode(nullptr), currentSize(0) {}

    // Destructor: Recorre la lista y libera cada nodo para evitar memory leaks
    ~LinkedStack() override {
        while (!isEmpty()) {
            pop(); // Reutilizamos pop() que ya maneja la lógica de borrado 'delete'
        }
    }

    void push(const T& element) override {
        // Asignación dinámica en el heap (puede lanzar std::bad_alloc si no hay memoria)
        topNode = new Node(element, topNode); 
        currentSize++;
    }

    T pop() override {
        if (isEmpty()) {
            throw std::underflow_error("Stack Underflow: Operacion POP en pila vacia.");
        }
        
        Node* nodeToDelete = topNode;       // Aislar el nodo
        T poppedValue = nodeToDelete->data; // Extraer el dato
        
        topNode = topNode->next;            // Reasignar el puntero principal
        delete nodeToDelete;                // Liberar la memoria al SO
        currentSize--;
        
        return poppedValue;
    }

    T top() const override {
        if (isEmpty()) {
            throw std::underflow_error("Pila vacia: No hay elementos en la cima.");
        }
        return topNode->data;
    }

    bool isEmpty() const override {
        return topNode == nullptr;
    }

    size_t getSize() const override {
        return currentSize;
    }
};

En la ingeniería de software moderna y en bibliotecas estándar (como la STL de C++ con su std::vector), implementar una estructura dinámica respaldada por un arreglo es la técnica dominante debido a la localidad de caché espacial que las listas enlazadas destruyen.

A esta arquitectura se le conoce como ***Pila de Arreglo Dinámico (Dynamic Array Stack)***. 

Para lograr que un arreglo estático se comporte de manera "dinámica" sin límites artificiales, debemos implementar un algoritmo de redimensionamiento automático (Resizing).

La estrategia matemáticamente óptima es la duplicación geométrica: cuando el arreglo alcanza su capacidad máxima, creamos un nuevo arreglo con el doble de la capacidad original, copiamos los elementos y liberamos la memoria antigua.

In [5]:
#include <iostream>
#include <stdexcept>
#include <algorithm> // Para std::copy

// Se asume la existencia de la interfaz IStack<T>

template <typename T>
class DynamicArrayStack : public IStack<T> {
private:
    T* array;
    size_t capacity;
    int topIndex;

    /**
     * @brief Redimensiona el arreglo subyacente duplicando su capacidad.
     * Operación O(N) que se amortiza a O(1) a largo plazo.
     */
    void resize() {
        size_t newCapacity = capacity * 2;
        T* newArray = new T[newCapacity];
        
        // Copiar elementos al nuevo bloque de memoria
        // std::copy es más eficiente que un bucle for tradicional
        std::copy(array, array + capacity, newArray);
        
        // Liberar el arreglo viejo para evitar Memory Leaks
        delete[] array;
        
        // Actualizar punteros y estado
        array = newArray;
        capacity = newCapacity;
    }

public:
    explicit DynamicArrayStack(size_t initialCap = 2) : capacity(initialCap), topIndex(-1) {
        array = new T[capacity];
    }

    ~DynamicArrayStack() override {
        delete[] array;
    }

    void push(const T& element) override {
        // Si la pila está llena, ya no lanzamos excepción, la redimensionamos
        if (topIndex == static_cast<int>(capacity) - 1) {
            resize(); 
        }
        array[++topIndex] = element;
    }

    T pop() override {
        if (isEmpty()) {
            throw std::underflow_error("Stack Underflow: Pila vacia.");
        }
        return array[topIndex--];
        
        // Nota Avanzada: En sistemas con restricciones estrictas de memoria,
        // aquí se podría implementar un algoritmo de 'shrink' (reducción) 
        // si el tamaño cae por debajo del 25% de la capacidad.
    }

    T top() const override {
        if (isEmpty()) {
            throw std::underflow_error("Pila vacia.");
        }
        return array[topIndex];
    }

    bool isEmpty() const override { return topIndex == -1; }
    size_t getSize() const override { return topIndex + 1; }
};

In [6]:
#include <iostream>
#include <chrono>
#include <string>

// Se asume que ArrayStack, LinkedStack y DynamicArrayStack están definidas aquí.

/**
 * @brief Ejecuta una prueba de esfuerzo sobre una pila genérica.
 * @tparam Stack Tipo de la pila a evaluar.
 * @param stack Referencia a la instancia de la pila.
 * @param name Nombre de la estructura para el reporte.
 * @param elements Número de operaciones a realizar.
 */
template <typename Stack>
void runStackBenchmark(Stack& stack, const std::string& name, int elements) {
    // Iniciar temporizador de alta resolución
    auto start = std::chrono::high_resolution_clock::now();

    // 1. Fase de Inserción (Push)
    for (int i = 0; i < elements; ++i) {
        stack.push(i);
    }

    // 2. Fase de Extracción (Pop)
    for (int i = 0; i < elements; ++i) {
        stack.pop();
    }

    // Detener temporizador
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> elapsed = end - start;

    std::cout << "[Benchmark] " << name << ": \t" 
              << elapsed.count() << " ms (" 
              << elements << " elementos)\n";
}

void ejemplo_02() {
    const int N = 10'000'000; // 10 millones de elementos

    std::cout << "=== INICIANDO BENCHMARK DE ESTRUCTURAS LIFO ===\n";
    std::cout << "Carga de trabajo: " << N << " Push + " << N << " Pop\n\n";

    try {
        // 1. Pila Estática (Requiere pre-asignar la capacidad máxima)
        ArrayStack<int> staticStack(N);
        runStackBenchmark(staticStack, "ArrayStack (Estatico)    ", N);

        // 2. Pila de Arreglo Dinámico (Inicia con capacidad pequeña, se redimensiona)
        DynamicArrayStack<int> dynamicArrStack(2);
        runStackBenchmark(dynamicArrStack, "DynamicArrayStack (Amortizado)", N);

        // 3. Pila de Lista Enlazada (Asignación nodo por nodo)
        LinkedStack<int> linkedStack;
        runStackBenchmark(linkedStack, "LinkedStack (Dinamico)     ", N);

    } catch (const std::exception& e) {
        std::cerr << "Error critico durante la ejecucion: " << e.what() << '\n';
    }
}

ejemplo_02();

=== INICIANDO BENCHMARK DE ESTRUCTURAS LIFO ===
Carga de trabajo: 10000000 Push + 10000000 Pop

[Benchmark] ArrayStack (Estatico)    : 	118.793 ms (10000000 elementos)
[Benchmark] DynamicArrayStack (Amortizado): 	144.421 ms (10000000 elementos)
[Benchmark] LinkedStack (Dinamico)     : 	585.516 ms (10000000 elementos)


1. ***ArrayStack (El más rápido - Aprox. $100$ ms)***: Al tener la memoria pre-reservada, push y pop son meras sumas de punteros y escrituras en registros contiguos. Maximiza los aciertos en la memoria caché L1/L2 de la CPU (Cache Hits).

2. ***DynamicArrayStack (Cerca del estático - Aprox. $1.5X$ ms)***: A pesar de realizar copias masivas $\mathcal{O}(N)$ durante los redimensionamientos (aprox. 24 redimensionamientos para llegar a 10 millones), el tiempo amortizado se mantiene casi a la par del estático. La copia de bloques de memoria mediante instrucciones SIMD del procesador es colosalmente rápida.

3. ***LinkedStack (El más lento - Aprox. $5X$ a $10X$ ms)***: El Overhead del Sistema Operativo y la CPU. Llamar a new Node() 10 millones de veces implica 10 millones de interrupciones para buscar espacio libre en el Heap (fragmentación). Recorrer los nodos destruye la Localidad Espacial; la CPU no puede predecir qué bloque de RAM cargar en la caché (Cache Misses), obligando a la CPU a esperar ciclos de reloj masivos buscando datos en la memoria principal.

# Cola estática 
Si intentamos implementar una cola con un arreglo simple (moviendo un índice rear al insertar y un índice front al extraer), nos enfrentaremos a un grave fallo de diseño conocido como Falsa Saturación (False Overflow) o Degradación Espacial.A medida que se encolan y desencolan elementos, ambos punteros avanzan hacia el final del arreglo. Eventualmente, rear alcanzará la capacidad máxima y lanzará un error de Overflow, incluso si hay celdas vacías al principio del arreglo (dejadas por los elementos ya extraídos). 

Desplazar todos los elementos hacia la izquierda para compactar el arreglo tomaría un tiempo inaceptable de $\mathcal{O}(N)$.La ciencia de la computación resuelve esto con una topología elegante: ***La Cola Circular***.

## Diseño 

Para gobernar el comportamiento en anillo de la memoria contigua, la clase requiere mantener el control estricto de cinco variables fundamentales:

* `arreglo` ($T[\,]$): El bloque de memoria física contigua asignado dinámicamente. Aunque lógicamente lo tratamos como un círculo, físicamente sigue siendo lineal en la memoria RAM (del índice $0$ al $N-1$).

* `capacidad` ($N$): Un escalar que define el límite estricto de celdas reservadas en el arreglo. Es un invariante durante la vida útil de esta implementación.

* `frente` (frontIndex): Un puntero lógico (índice entero) que señala la posición en memoria del elemento más antiguo de la estructura. Es el punto exclusivo de extracción (dequeue). Se inicializa en $0$ y avanza circularmente mediante la ecuación:$frente = (frente + 1) \bmod capacidad$

* `final` (rearIndex): Un puntero lógico que señala la posición del último elemento insertado. Es el punto exclusivo de entrada (enqueue). Se inicializa en $-1$ para representar que no hay ninguna posición válida ocupada al arrancar. Avanza con:$final = (final + 1) \bmod capacidad$

* `cantidad` (count): La variable arquitectónica más crítica. Rastrear explícitamente el volumen de elementos ($0 \le cantidad \le capacidad$) es la técnica que nos permite determinar en tiempo asintótico $\mathcal{O}(1)$ si la estructura sufre un Underflow (cantidad == 0) o un Overflow (cantidad == capacidad). Sin esta variable auxiliar, la condición espacial de "cola vacía" y "cola llena" se volvería matemáticamente ambigua cuando los punteros frente y final colisionan en la misma coordenada del arreglo.

***Algoritmos***

```
Variables de Estado:
    arreglo, capacidad
    frente = 0, final = -1, cantidad = 0

 1 ENQUEUE(Cola, elemento):
 2   Si Cola.cantidad == Cola.capacidad entonces
 3       Lanzar Excepción "Queue Overflow"
 4   
 5   // El operador módulo hace que el índice 'dé la vuelta' al llegar al límite
 6   Cola.final = (Cola.final + 1) MOD Cola.capacidad
 7   Cola.arreglo[Cola.final] = elemento
 8   Cola.cantidad = Cola.cantidad + 1
```

```
 1 DEQUEUE(Cola):
 2   Si Cola.cantidad == 0 entonces
 3       Lanzar Excepción "Queue Underflow"
 4   
 5   valor_extraido = Cola.arreglo[Cola.frente]
 6   Cola.frente = (Cola.frente + 1) MOD Cola.capacidad
 7   Cola.cantidad = Cola.cantidad - 1
 8   
 9   Retornar valor_extraido
```

## Abstacción de colas en C++ 

``` cpp
template <typename T>
class IQueue {
public:
    virtual ~IQueue() = default;

    virtual void enqueue(const T& element) = 0; // Insertar al final (Rear) O(1)
    virtual T dequeue() = 0;                    // Extraer del frente (Front) O(1)
    virtual T front() const = 0;                // Consultar el frente O(1)
    
    virtual bool isEmpty() const = 0;
    virtual size_t getSize() const = 0;
};

In [7]:
/**
 * @brief Interfaz Abstracta para una Cola (Queue).
 * Define el contrato estricto FIFO (First-In, First-Out).
 * @tparam T Tipo de dato contenido en la estructura.
 */
template <typename T>
class IQueue {
public:
    virtual ~IQueue() = default;

    virtual void enqueue(const T& element) = 0; // Insertar al final (Rear) O(1)
    virtual T dequeue() = 0;                    // Extraer del frente (Front) O(1)
    virtual T front() const = 0;                // Consultar el frente O(1)
    
    virtual bool isEmpty() const = 0;
    virtual size_t getSize() const = 0;
};

In [8]:
#include <iostream>
#include <stdexcept>

// Se asume la existencia de la interfaz IQueue<T>

/**
 * @brief Implementación estática de una Cola Circular (Circular Queue).
 * * Utiliza un arreglo de capacidad fija y aritmética modular para reutilizar 
 * el espacio de memoria, resolviendo el problema de la falsa saturación 
 * (degradación espacial) inherente a las colas lineales estáticas.
 * Todas las operaciones fundamentales operan en tiempo estricto O(1).
 * * @tparam T Tipo de dato almacenado en la cola.
 */
template <typename T>
class CircularQueue : public IQueue<T> {
private:
    T* array;           ///< Puntero al bloque de memoria física contigua.
    size_t capacity;    ///< Límite máximo inmutable de celdas reservadas.
    int frontIndex;     ///< Índice lógico que señala el punto de extracción (frente).
    int rearIndex;      ///< Índice lógico que señala el punto de inserción (final).
    size_t count;       ///< Variable de estado crítica que rastrea el volumen de datos actual.

public:
    /**
     * @brief Constructor explícito de la Cola Circular.
     * Asigna dinámicamente el bloque de memoria contigua en el Heap.
     * @param cap Capacidad máxima de la cola (por defecto 10).
     */
    explicit CircularQueue(size_t cap = 10) 
        : capacity(cap), frontIndex(0), rearIndex(-1), count(0) {
        array = new T[capacity];
    }

    /**
     * @brief Destructor.
     * Garantiza la liberación de la memoria física contigua, previniendo memory leaks.
     */
    ~CircularQueue() override {
        delete[] array;
    }

    /**
     * @brief Encola un elemento en la parte posterior de la estructura.
     * Utiliza el operador módulo (%) para calcular el índice circular, garantizando
     * que el puntero 'dé la vuelta' si alcanza el final físico del arreglo.
     * * @param element Referencia constante al elemento a insertar.
     * @throws std::overflow_error Si el volumen de datos (count) alcanza la capacidad máxima.
     */
    void enqueue(const T& element) override {
        if (count == capacity) {
            throw std::overflow_error("Queue Overflow: La cola circular esta llena.");
        }
        // Aritmética modular para calcular el siguiente índice disponible O(1)
        rearIndex = (rearIndex + 1) % capacity;
        array[rearIndex] = element;
        count++;
    }

    /**
     * @brief Desencola y retorna el elemento en el frente de la estructura.
     * Avanza el índice frontal utilizando aritmética modular para mantener el 
     * comportamiento en anillo.
     * * @return El elemento extraído del frente.
     * @throws std::underflow_error Si se intenta extraer de una estructura sin elementos.
     */
    T dequeue() override {
        if (isEmpty()) {
            throw std::underflow_error("Queue Underflow: La cola esta vacia.");
        }
        
        T dequeuedValue = array[frontIndex];
        // Aritmética modular para avanzar el frente dando la vuelta si es necesario O(1)
        frontIndex = (frontIndex + 1) % capacity;
        count--;
        
        return dequeuedValue;
    }

    /**
     * @brief Inspecciona el elemento en el frente sin alterar el estado de la cola.
     * @return Copia del elemento frontal.
     * @throws std::underflow_error Si la cola está vacía.
     */
    T front() const override {
        if (isEmpty()) throw std::underflow_error("Cola vacia.");
        return array[frontIndex];
    }

    /**
     * @brief Consulta el estado de vaciado de la estructura.
     * @return true si el contador de elementos es estrictamente 0, false en caso contrario.
     */
    bool isEmpty() const override { 
        return count == 0; 
    }

    /**
     * @brief Retorna el volumen exacto de elementos almacenados actualmente.
     * @return Cantidad de elementos (size_t).
     */
    size_t getSize() const override { 
        return count; 
    }
};

# Cola dinámica 

Al igual que con las pilas, si necesitamos una cola sin límite estricto de capacidad y queremos evitar el redimensionamiento del arreglo, empleamos una arquitectura de nodos distribuidos en el Heap.

Para garantizar que tanto enqueue (insertar al final) como dequeue (extraer del frente) operen en tiempo estricto de $\mathcal{O}(1)$, nuestra lista enlazada debe mantener dos punteros: head (cabeza/frente) y tail (cola/final). Iterar desde la cabeza hasta el final para insertar arruinaría la complejidad a $\mathcal{O}(N)$.

***Algoritmos***

```
 1 ENQUEUE(Cola, elemento):
 2   nuevo_nodo = crear_nodo(elemento)
 3   
 4   Si Cola.esta_vacia() entonces
 5       Cola.frente = nuevo_nodo
 6       Cola.final = nuevo_nodo
 7   Sino
 8       Cola.final.siguiente = nuevo_nodo
 9       Cola.final = nuevo_nodo
```

```
 1 DEQUEUE(Cola):
 2   Si Cola.esta_vacia() entonces Lanzar Excepción
 3   
 4   nodo_a_eliminar = Cola.frente
 5   valor = nodo_a_eliminar.valor
 6   
 7   Cola.frente = Cola.frente.siguiente
 8   
 9   // Prevención de punteros colgantes (Dangling Pointers)
10    Si Cola.frente es NULO entonces
11        Cola.final = NULO
12        
13    liberar_memoria(nodo_a_eliminar)
14    Retornar valor
```

In [9]:
#include <iostream>
#include <stdexcept>

/**
 * @brief Implementación dinámica de Cola mediante nodos distribuidos en el Heap.
 * Garantiza O(1) estricto sin redimensionamientos, a costa de la localidad de caché.
 * @tparam T Tipo de dato almacenado.
 */
template <typename T>
class LinkedQueue : public IQueue<T> {
private:
    /**
     * @brief Estructura interna del nodo.
     */
    struct Node {
        T data;     ///< Valor almacenado.
        Node* next; ///< Puntero al siguiente nodo en la secuencia lógica.
        
        /**
         * @brief Constructor del nodo.
         * @param val Valor a inicializar.
         */
        Node(const T& val) : data(val), next(nullptr) {}
    };

    Node* head; ///< Puntero al frente de la cola (extracción).
    Node* tail; ///< Puntero al final de la cola (inserción).
    size_t currentSize; ///< Variable de estado para resolución de tamaño en O(1).

public:
    /**
     * @brief Constructor. Inicializa una cola vacía.
     */
    LinkedQueue() : head(nullptr), tail(nullptr), currentSize(0) {}

    /**
     * @brief Destructor. Previene memory leaks iterando y liberando cada nodo.
     */
    ~LinkedQueue() override {
        while (!isEmpty()) {
            dequeue();
        }
    }

    /**
     * @brief Encola un elemento al final de la lista.
     * @param element Referencia al dato.
     * @throws std::bad_alloc Si el sistema operativo deniega la memoria en el Heap.
     */
    void enqueue(const T& element) override {
        Node* newNode = new Node(element);
        if (isEmpty()) {
            head = tail = newNode;
        } else {
            tail->next = newNode;
            tail = newNode;
        }
        currentSize++;
    }

    /**
     * @brief Desencola y destruye el nodo frontal.
     * @return El dato extraído.
     * @throws std::underflow_error Si la cola está vacía.
     */
    T dequeue() override {
        if (isEmpty()) {
            throw std::underflow_error("Queue Underflow.");
        }
        Node* nodeToDelete = head;
        T dequeuedValue = nodeToDelete->data;
        
        head = head->next;
        if (head == nullptr) {
            tail = nullptr; // Prevención de Dangling Pointer
        }
        
        delete nodeToDelete;
        currentSize--;
        return dequeuedValue;
    }

    /**
     * @brief Consulta el valor en el nodo 'head'.
     * @return Valor del frente.
     * @throws std::underflow_error Si la cola está vacía.
     */
    T front() const override {
        if (isEmpty()) throw std::underflow_error("Cola vacia.");
        return head->data;
    }

    /** @brief Verifica si el puntero 'head' es nulo. */
    bool isEmpty() const override { return head == nullptr; }
    
    /** @brief Retorna el volumen actual de nodos. */
    size_t getSize() const override { return currentSize; }
};

Al igual que el en caso de Pilas, tambien se puede hacer una implementación empleando arreglos 

In [10]:
/**
 * @brief Implementación de Cola Circular con memoria dinámica redimensionable.
 * Mantiene tiempo O(1) amortizado para operaciones FIFO previniendo la degradación espacial.
 * @tparam T Tipo de dato almacenado.
 */
template <typename T>
class DynamicCircularQueue : public IQueue<T> {
private:
    T* array;           ///< Puntero al bloque de memoria física contigua.
    size_t capacity;    ///< Límite actual de celdas reservadas.
    int frontIndex;     ///< Índice lógico de extracción.
    int rearIndex;      ///< Índice lógico de inserción.
    size_t count;       ///< Rastreador estricto del volumen de datos.

    /**
     * @brief Redimensiona y "desenrolla" la cola circular.
     * Duplica la capacidad geométrica y mapea el orden lógico al nuevo orden lineal físico.
     * Complejidad: O(N) en tiempo, O(1) amortizado sobre N operaciones.
     */
    void resize() {
        size_t newCapacity = capacity * 2;
        T* newArray = new T[newCapacity];
        
        for (size_t i = 0; i < count; ++i) {
            int oldPhysicalIndex = (frontIndex + i) % capacity;
            newArray[i] = array[oldPhysicalIndex];
        }
        
        delete[] array;
        array = newArray;
        capacity = newCapacity;
        
        frontIndex = 0;
        rearIndex = count - 1; 
    }

public:
    /**
     * @brief Constructor explícito.
     * @param initialCap Capacidad inicial del arreglo (por defecto 4).
     */
    explicit DynamicCircularQueue(size_t initialCap = 4) 
        : capacity(initialCap), frontIndex(0), rearIndex(-1), count(0) {
        array = new T[capacity];
    }

    /**
     * @brief Destructor. Libera la memoria contigua asignada al arreglo.
     */
    ~DynamicCircularQueue() override {
        delete[] array;
    }

    /**
     * @brief Inserta un elemento. Ejecuta redimensionamiento si se alcanza la capacidad máxima.
     * @param element Referencia al dato a insertar.
     */
    void enqueue(const T& element) override {
        if (count == capacity) {
            resize();
        }
        rearIndex = (rearIndex + 1) % capacity;
        array[rearIndex] = element;
        count++;
    }

    /**
     * @brief Extrae el dato frontal.
     * @return El valor del dato en el frente.
     * @throws std::underflow_error Si se intenta extraer de una cola vacía.
     */
    T dequeue() override {
        if (isEmpty()) {
            throw std::underflow_error("Queue Underflow.");
        }
        T dequeuedValue = array[frontIndex];
        frontIndex = (frontIndex + 1) % capacity;
        count--;
        return dequeuedValue;
    }

    /**
     * @brief Inspecciona el frente de la cola en O(1).
     * @return El valor del dato frontal.
     * @throws std::underflow_error Si la cola está vacía.
     */
    T front() const override {
        if (isEmpty()) throw std::underflow_error("Cola vacia.");
        return array[frontIndex];
    }

    /** @brief Retorna true si count == 0. */
    bool isEmpty() const override { return count == 0; }
    
    /** @brief Retorna el valor actual de count. */
    size_t getSize() const override { return count; }
};

In [11]:
#include <iomanip>

/**
 * @brief Ejecuta una prueba de estrés sobre una implementación genérica de Cola.
 * @tparam Queue Tipo de la estructura de datos FIFO.
 * @param queue Referencia a la instancia a evaluar.
 * @param name Identificador de la estructura para el reporte en consola.
 * @param elements Magnitud de la carga de trabajo (N).
 */
template <typename Queue>
void runQueueBenchmark(Queue& queue, const std::string& name, int elements) {
    // Sincronización de reloj de alta resolución
    auto start = std::chrono::high_resolution_clock::now();

    // Fase 1: Saturación (Enqueue)
    for (int i = 0; i < elements; ++i) {
        queue.enqueue(i);
    }

    // Fase 2: Vaciado (Dequeue)
    for (int i = 0; i < elements; ++i) {
        queue.dequeue();
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> elapsed = end - start;

    std::cout << std::left << std::setw(35) << name
              << ": " << std::fixed << std::setprecision(2) 
              << elapsed.count() << " ms\n";
}

void ejemplo_03() {
    // 10 millones de operaciones para forzar el caché L3 y la memoria principal
    const int N = 10'000'000; 

    std::cout << "=== BENCHMARK DE ARQUITECTURAS FIFO ===\n";
    std::cout << "Carga: " << N << " Enqueue + " << N << " Dequeue\n";
    std::cout << std::string(55, '-') << "\n";

    try {
        // 1. Cola Circular Estática: Requiere reserva total anticipada O(N) espacial
        CircularQueue<int> staticQueue(N);
        runQueueBenchmark(staticQueue, "[Estática] CircularQueue", N);

        // 2. Cola Circular Dinámica: Inicia pequeña (ej. 4) y escala geométricamente
        DynamicCircularQueue<int> dynamicQueue(4);
        runQueueBenchmark(dynamicQueue, "[Dinámica] DynamicCircularQueue", N);

        // 3. Cola de Nodos: Asignación dinámica individual en el Heap
        LinkedQueue<int> linkedQueue;
        runQueueBenchmark(linkedQueue, "[Nodos] LinkedQueue", N);

    } catch (const std::exception& e) {
        std::cerr << "Excepción fatal durante perfilamiento: " << e.what() << '\n';
    }
}

ejemplo_03();

=== BENCHMARK DE ARQUITECTURAS FIFO ===
Carga: 10000000 Enqueue + 10000000 Dequeue
-------------------------------------------------------
[Estática] CircularQueue          : 191.54 ms
[Dinámica] DynamicCircularQueue   : 322.08 ms
[Nodos] LinkedQueue                : 601.21 ms


1. ***CircularQueue*** (La más rápida - 191.54 ms): Al operar sobre un bloque de memoria física pre-reservada y contigua, las operaciones enqueue y dequeue se reducen a manipulación aritmética directa (avanzar índices de punteros y evaluar el operador módulo %) junto con asignaciones directas en memoria. Esta arquitectura maximiza de manera absoluta la localidad espacial, lo que se traduce en un índice altísimo de aciertos en la memoria caché L1/L2 de la CPU (Cache Hits).

2. ***DynamicCircularQueue*** (Rendimiento amortizado competitivo - 322.08 ms, Aprox. $1.6X$ más lenta que la estática): A pesar de la penalización asintótica de las copias masivas $\mathcal{O}(N)$ y el sobrecosto algorítmico de "desenrollar" (unwrap) el anillo lógico durante cada redimensionamiento, el tiempo de ejecución se mantiene notablemente veloz. Esto demuestra físicamente el poder del Análisis Amortizado: las expansiones geométricas son infrecuentes (aprox. 24 expansiones para alcanzar 10 millones) y la transferencia de bloques de memoria contigua se ejecuta a velocidades colosales gracias a las instrucciones SIMD (Single Instruction, Multiple Data) del procesador.

3. ***LinkedQueue*** (La más lenta - 601.21 ms, Aprox. $3X$ más lenta que la estática): El rendimiento se degrada drásticamente debido a la sobrecarga (Overhead) del Sistema Operativo y la arquitectura del procesador.Invocar `new Node()` y `delete` 10 millones de veces exige intervenciones masivas del gestor del Heap para encontrar y liberar fragmentos de memoria, provocando contención severa.Al asignar nodos de forma distribuida, se destruye por completo la Localidad Espacial. El predictor de la CPU no puede anticipar qué dirección de memoria cargar a continuación (Cache Misses frecuentes), obligando al procesador a detenerse durante cientos de ciclos de reloj esperando que los datos viajen desde la latente memoria RAM principal.

# La Biblioteca Estándar (STL): Adaptadores de Contenedores
En C++ moderno, `std::stack` y `std::queue` no son contenedores en sí mismos, sino adaptadores de contenedores (Container Adapters). Esto significa que envuelven una estructura subyacente (por defecto, std::deque o cola doblemente terminada) y restringen su interfaz para forzar matemáticamente el comportamiento LIFO o FIFO.

Este sistema combina ambas arquitecturas: una Cola para procesar operaciones entrantes (FIFO) y una Pila para mantener un historial de reversión o Undo (LIFO).

In [12]:
#include <iostream>
#include <stack>
#include <queue>
#include <string>

/**
 * @brief Sistema procesador de transacciones con capacidad de reversión (Undo).
 * Integra std::queue (FIFO) para el flujo de trabajo y std::stack (LIFO) para el historial.
 */
class TransactionProcessor {
private:
    std::queue<std::string> pendingTasks; // Cola de tareas por procesar
    std::stack<std::string> taskHistory;  // Pila de tareas completadas para 'Undo'

public:
    /**
     * @brief Encola una nueva tarea en O(1).
     */
    void addTask(const std::string& task) {
        pendingTasks.push(task); // std::queue usa 'push' en lugar de 'enqueue'
        std::cout << "[Ingreso] Tarea encolada: " << task << "\n";
    }

    /**
     * @brief Procesa la tarea más antigua (FIFO) y la apila en el historial (LIFO).
     */
    void processNext() {
        if (pendingTasks.empty()) {
            std::cout << "[Aviso] No hay tareas pendientes.\n";
            return;
        }

        // 1. Extraer del frente de la cola (FIFO)
        std::string currentTask = pendingTasks.front();
        pendingTasks.pop(); // std::queue usa 'pop' en lugar de 'dequeue'

        std::cout << "[Ejecución] Procesando: " << currentTask << "\n";

        // 2. Apilar en el historial (LIFO)
        taskHistory.push(currentTask);
    }

    /**
     * @brief Revierte la última tarea procesada extrayéndola de la cima de la pila.
     */
    void undoLast() {
        if (taskHistory.empty()) {
            std::cout << "[Error] El historial esta vacio. Nada que revertir.\n";
            return;
        }

        // 1. Extraer de la cima de la pila (LIFO)
        std::string lastTask = taskHistory.top();
        taskHistory.pop();

        std::cout << "[Reversión] Deshaciendo tarea: " << lastTask << "\n";
    }
};

void ejemplo_04() {
    TransactionProcessor editorEngine;

    std::cout << "=== FASE 1: RECEPCION DE EVENTOS DEL USUARIO (FIFO) ===\n";
    // El usuario realiza acciones rápidas; el sistema las encola para procesarlas sin bloquear la interfaz.
    editorEngine.addTask("Escribir_Parrafo_1");
    editorEngine.addTask("Aplicar_Formato_Negrita");
    editorEngine.addTask("Insertar_Imagen_Logo");

    std::cout << "\n=== FASE 2: PROCESAMIENTO SECUENCIAL ===\n";
    // El motor en segundo plano comienza a ejecutar (FIFO garantiza el orden cronológico de ejecución)
    editorEngine.processNext(); 
    editorEngine.processNext(); 

    std::cout << "\n=== FASE 3: INTERRUPCION Y NUEVA TAREA ===\n";
    // Mientras procesa, entra una nueva tarea. Se va al final de la cola, respetando a las anteriores.
    editorEngine.addTask("Cambiar_Fuente_Arial"); 

    std::cout << "\n=== FASE 4: REVERSION DE ESTADO (LIFO - Operacion Ctrl+Z) ===\n";
    // El usuario se equivoca y presiona Ctrl+Z dos veces.
    // La pila interviene y fuerza a deshacer de lo más reciente a lo más antiguo.
    editorEngine.undoLast(); // Revierte: Aplicar_Formato_Negrita (Lo último ejecutado)
    editorEngine.undoLast(); // Revierte: Escribir_Parrafo_1 (La acción original)

    std::cout << "\n=== FASE 5: RETOMAR EL PROCESAMIENTO ===\n";
    // El sistema continúa con lo que había quedado pendiente en la cola
    editorEngine.processNext(); // Ejecuta: Insertar_Imagen_Logo
}

ejemplo_04();

=== FASE 1: RECEPCION DE EVENTOS DEL USUARIO (FIFO) ===
[Ingreso] Tarea encolada: Escribir_Parrafo_1
[Ingreso] Tarea encolada: Aplicar_Formato_Negrita
[Ingreso] Tarea encolada: Insertar_Imagen_Logo

=== FASE 2: PROCESAMIENTO SECUENCIAL ===
[Ejecución] Procesando: Escribir_Parrafo_1
[Ejecución] Procesando: Aplicar_Formato_Negrita

=== FASE 3: INTERRUPCION Y NUEVA TAREA ===
[Ingreso] Tarea encolada: Cambiar_Fuente_Arial

=== FASE 4: REVERSION DE ESTADO (LIFO - Operacion Ctrl+Z) ===
[Reversión] Deshaciendo tarea: Aplicar_Formato_Negrita
[Reversión] Deshaciendo tarea: Escribir_Parrafo_1

=== FASE 5: RETOMAR EL PROCESAMIENTO ===
[Ejecución] Procesando: Insertar_Imagen_Logo
